# LangChain agent observability: compare cost and latency

Requires Elasticsearch and Kibana 9.5+ on Elastic Cloud with the Managed OTLP endpoint, and an
[OpenRouter](https://openrouter.ai/) API key (the full run costs a few US cents). Copy `.env.example`
to `.env` and set an API key created in Kibana **Dev Tools**:

```
POST /_security/api_key
{ "name": "langchain-agent-observability-notebook", "role_descriptors": {} }
```

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

In [ ]:
import json
import os
import re
import time

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch, helpers

load_dotenv()

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
OTLP_ENDPOINT = os.environ["OTLP_ENDPOINT"].rstrip("/")
OTLP_API_KEY = os.getenv("OTLP_API_KEY") or API_KEY
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
SMALL_MODEL = os.getenv("SMALL_MODEL", "openai/gpt-5.6-luna")
LARGE_MODEL = os.getenv("LARGE_MODEL", "openai/gpt-5.6-sol")
INFERENCE_ID = os.getenv("INFERENCE_ID", ".elser-2-elastic")
RUNSET = os.getenv("LAB_RUNSET", "shopping-model-swap")
REPEATS = int(os.getenv("LAB_REPEATS", "3"))

SERVICE = "shop-assistant"
LAB_SOURCE = "langchain-agent-observability-notebook"  # resource attribute, Clean up deletes only these spans
CATALOG = "shop-assistant-catalog"
TRACES = "traces-generic.otel-default"

# The notebook's own Elasticsearch calls stay out of the traces.
os.environ["OTEL_PYTHON_INSTRUMENTATION_ELASTICSEARCH_ENABLED"] = "false"
es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)


def esql(query, title=None):
    if title:
        print(f"\n{title}")
    print(es.esql.query(query=query, format="txt").body)


def esql_rows(query):
    response = es.esql.query(query=query)
    names = [c["name"] for c in response["columns"]]
    return [dict(zip(names, row)) for row in response["values"]]


print(f"elasticsearch {es.info()['version']['number']}")

## 2. Load the sample store

Six coffee makers and two policies in one index. `content` is a `semantic_text` field; the other fields keep the exact specifications.

In [ ]:
PRODUCTS = [
    {
        "name": "Compact Brew",
        "price_usd": 79,
        "width_cm": 18,
        "cups": 4,
        "programmable_timer": False,
        "features": "Glass carafe, auto shut-off after 30 minutes",
    },
    {
        "name": "Morning Mini",
        "price_usd": 95,
        "width_cm": 19,
        "cups": 6,
        "programmable_timer": True,
        "features": "Glass carafe, 24-hour programmable timer",
    },
    {
        "name": "Family Brew",
        "price_usd": 89,
        "width_cm": 24,
        "cups": 12,
        "programmable_timer": True,
        "features": "Glass carafe, programmable timer, brew strength control",
    },
    {
        "name": "Slim Pro",
        "price_usd": 129,
        "width_cm": 17,
        "cups": 8,
        "programmable_timer": True,
        "features": "Thermal carafe, programmable timer",
    },
    {
        "name": "Travel Cup",
        "price_usd": 49,
        "width_cm": 12,
        "cups": 1,
        "programmable_timer": False,
        "features": "Single-serve brewer that fills a travel mug",
    },
    {
        "name": "Counter Fit",
        "price_usd": 99,
        "width_cm": 20,
        "cups": 5,
        "programmable_timer": False,
        "features": "Thermal carafe keeps coffee hot for 2 hours",
    },
]
POLICIES = [
    {
        "title": "Returns",
        "text": "Unused coffee makers can be returned within 30 days of delivery for a full refund. "
        "Opened coffee makers can be returned within 30 days if all parts are included; a 10% "
        "restocking fee applies. Return shipping is free with the prepaid label.",
    },
    {
        "title": "Shipping",
        "text": "Standard shipping is free on orders over $50 and takes 3 to 5 business days. Express "
        "shipping costs $12 and takes 1 to 2 business days. Orders placed before 2 p.m. ship "
        "the same day.",
    },
]


def product_text(p):
    timer = (
        "has a programmable timer"
        if p["programmable_timer"]
        else "has no programmable timer"
    )
    return (
        f"{p['name']} coffee maker: ${p['price_usd']}, {p['width_cm']} cm wide, brews "
        f"{p['cups']} cups, {timer}. {p['features']}."
    )


es.options(ignore_status=404).indices.delete(index=CATALOG)
es.indices.create(
    index=CATALOG,
    mappings={
        "properties": {
            "kind": {"type": "keyword"},
            "content": {"type": "semantic_text", "inference_id": INFERENCE_ID},
            "name": {"type": "keyword"},
            "title": {"type": "keyword"},
            "price_usd": {"type": "float"},
            "width_cm": {"type": "float"},
            "cups": {"type": "integer"},
            "programmable_timer": {"type": "boolean"},
            "features": {"type": "text"},
            "text": {"type": "text"},
        }
    },
)

docs = [{"kind": "product", "content": product_text(p), **p} for p in PRODUCTS]
docs += [
    {"kind": "policy", "content": f"{p['title']} policy: {p['text']}", **p}
    for p in POLICIES
]
for attempt in range(
    5
):  # the first semantic_text call can wait for the inference endpoint
    try:
        _, errors = helpers.bulk(
            es,
            ({"_index": CATALOG, "_source": d} for d in docs),
            refresh="wait_for",
            raise_on_error=False,
        )
        if not errors:
            break
    except Exception as exc:
        errors = [exc]
    print(f"attempt {attempt + 1} failed, retrying in 20 s: {errors[0]}")
    time.sleep(20)

print(f"{CATALOG}: {es.count(index=CATALOG)['count']} records")

## 3. Record the agent's work with OpenTelemetry

Message capture off (`NO_CONTENT`), instrumentor enabled before the agent is created. The assistant's Elasticsearch client is traced, so each search appears under its retrieval span.

In [ ]:
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = OTLP_ENDPOINT
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=ApiKey%20{OTLP_API_KEY}"
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "NO_CONTENT"

from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.instrumentation.genai.langchain import LangChainInstrumentor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.trace import Status, StatusCode

provider = TracerProvider(
    resource=Resource.create(
        {
            "service.name": SERVICE,
            "lab.source": LAB_SOURCE,
        }
    )
)
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
trace.set_tracer_provider(provider)
LangChainInstrumentor().instrument()
tracer = trace.get_tracer(SERVICE)

os.environ["OTEL_PYTHON_INSTRUMENTATION_ELASTICSEARCH_ENABLED"] = "true"
shop_es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=60)

## 4. Build the shopping assistant

`build_agent` is the code from the article. `UsageCollector` keeps each call's `response_metadata["cost"]` and token usage; `run_request` wraps one request in the `shop.request` root span with the `app.*` attributes. Temperature 0, 1,000 output tokens max, retries off.

In [ ]:
from langchain.agents import create_agent
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openrouter import ChatOpenRouter


class UsageCollector(BaseCallbackHandler):
    def __init__(self):
        self.calls, self._requested = [], {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        params, metadata = (
            kwargs.get("invocation_params") or {},
            kwargs.get("metadata") or {},
        )
        self._requested[run_id] = (
            params.get("model")
            or params.get("model_name")
            or metadata.get("ls_model_name")
        )

    def on_llm_end(self, response, *, run_id, **kwargs):
        for generation in response.generations[0]:
            message = generation.message
            self.calls.append(
                {
                    "model": self._requested.pop(run_id, None)
                    or message.response_metadata.get("model_name"),
                    "metadata": dict(message.response_metadata),
                    "usage": dict(message.usage_metadata or {}),
                }
            )


usage = UsageCollector()


def text_of(message):
    text = getattr(message, "text", None)
    if isinstance(text, str):
        return str(text)
    return text() if callable(text) else str(message.content)


def model(name):
    return ChatOpenRouter(
        model=name,
        api_key=OPENROUTER_API_KEY,
        temperature=0,
        max_tokens=1000,
        max_retries=0,
        callbacks=[usage],
    )


SHOP_INSTRUCTIONS = (
    "You are the shopping assistant of an online coffee maker store. For questions about "
    "returns or shipping, call answer_store_question. For questions about choosing or comparing "
    "coffee makers, call compare_products. Call exactly one tool, passing the customer's "
    "question unchanged, then give the tool's answer to the customer without adding, removing or "
    "changing any product, price, specification or policy detail."
)
POLICY_INSTRUCTIONS = (
    "Answer the customer's question using only the store policy records provided. Quote time "
    "limits and costs exactly. Answer in at most three sentences."
)
COMPARE_INSTRUCTIONS = (
    "You compare coffee makers for a store customer using only the product records provided. "
    "List every product that meets all of the customer's requirements with its name, price, "
    "width, cups and whether it has a programmable timer, then say in one or two sentences how "
    "they differ. If no product meets every requirement, start the answer with exactly: "
    "'No product in the catalog meets all requirements.' If the customer asks for information "
    "the records do not contain, say 'The catalog does not include' followed by what is missing. "
    "Never invent specifications."
)
RETRIEVE_SPAN = {"product": "retrieve products", "policy": "retrieve policies"}
FIELDS = {
    "product": [
        "name",
        "price_usd",
        "width_cm",
        "cups",
        "programmable_timer",
        "features",
    ],
    "policy": ["title", "text"],
}


def retrieve(question, kind, size):
    with tracer.start_as_current_span(RETRIEVE_SPAN[kind]):
        response = shop_es.search(
            index=CATALOG,
            size=size,
            source=FIELDS[kind],
            query={
                "bool": {
                    "must": {"semantic": {"field": "content", "query": question}},
                    "filter": {"term": {"kind": kind}},
                }
            },
        )
    return [hit["_source"] for hit in response["hits"]["hits"]]


def answer(question, kind, size, model_name):
    records = retrieve(question, kind, size)
    instructions = COMPARE_INSTRUCTIONS if kind == "product" else POLICY_INSTRUCTIONS
    reply = model(model_name).invoke(
        [
            SystemMessage(content=instructions),
            HumanMessage(
                content=f"Customer question: {question}\n\nRecords:\n{json.dumps(records, indent=2)}"
            ),
        ]
    )
    return text_of(reply)


def build_agent(comparison_model=LARGE_MODEL):
    @tool
    def answer_store_question(question: str) -> str:
        """Answer questions about returns or shipping policies."""
        return answer(question, "policy", size=2, model_name=SMALL_MODEL)

    @tool
    def compare_products(question: str) -> str:
        """Compare coffee makers with the customer's requirements."""
        return answer(question, "product", size=6, model_name=comparison_model)

    return create_agent(
        model=model(SMALL_MODEL),
        tools=[answer_store_question, compare_products],
        name="shop_assistant",
        system_prompt=SHOP_INSTRUCTIONS,
    )


AGENTS = {"baseline": build_agent(LARGE_MODEL), "candidate": build_agent(SMALL_MODEL)}

In [ ]:
CASES = {
    "compact": {
        "question": "Compare all coffee makers under $100, no wider than 20 cm, that brew at least 4 cups. Which options fit?",
        "expect": ["Compact Brew", "Morning Mini", "Counter Fit"],
    },
    "timer": {
        "question": "I want coffee ready when I wake up. Which coffee makers have a programmable timer, and which one is the cheapest?",
        "expect": ["Morning Mini", "Family Brew", "Slim Pro"],
    },
    "family": {
        "question": "We are a family of five and brew about 10 cups every morning. Which coffee makers make at least 10 cups at once?",
        "expect": ["Family Brew"],
    },
    "no_match": {
        "question": "Is there a coffee maker under $60 that brews at least 6 cups?",
        "phrase": "no product in the catalog meets",
    },
    "unknown": {
        "question": "Which coffee maker under $100 is the quietest? I need the noise level in decibels.",
        "phrase": "catalog does not include",
        "forbid": r"\d+\s*(db|decibel)",
    },
}


def check(case, text):
    spec, lowered = CASES[case], text.lower()
    ok = all(name.lower() in lowered for name in spec.get("expect", []))
    if "phrase" in spec:
        ok = ok and spec["phrase"] in lowered
    if "forbid" in spec:
        ok = ok and not re.search(spec["forbid"], lowered)
    return ok


def run_request(question, variant, case="", runset=RUNSET):
    usage.calls.clear()
    started = time.perf_counter()
    with tracer.start_as_current_span("shop.request") as span:
        span.set_attribute("app.runset", runset)
        span.set_attribute("app.case", case)
        span.set_attribute("app.variant", variant)
        try:
            result = AGENTS[variant].invoke(
                {"messages": [{"role": "user", "content": question}]},
                config={
                    "metadata": {"agent_name": "shop_assistant"}
                },  # recorded as invoke_agent
            )
            text, error = text_of(result["messages"][-1]), None
        except Exception as exc:
            text, error = "", f"{type(exc).__name__}: {exc}"
            span.record_exception(exc)
            span.set_status(Status(StatusCode.ERROR, error[:200]))
        cost = sum(call["metadata"].get("cost", 0.0) for call in usage.calls)
        passed = bool(case) and not error and check(case, text)
        span.set_attribute("app.check_passed", passed)
        span.set_attribute("app.model_cost.usd", cost)
        trace_id = format(span.get_span_context().trace_id, "032x")
    return {
        "trace_id": trace_id,
        "case": case,
        "variant": variant,
        "answer": text,
        "error": error,
        "passed": passed,
        "seconds": time.perf_counter() - started,
        "cost": cost,
        "calls": [dict(call) for call in usage.calls],
    }


policy_demo = run_request(
    "How long do I have to return a coffee maker I already opened?",
    "baseline",
    runset="shopping-demo",
)
print(policy_demo["answer"], "\n")
demo = run_request(
    CASES["compact"]["question"], "baseline", case="compact", runset="shopping-demo"
)
print(demo["answer"])
print(
    f"\n{demo['seconds']:.2f} s | check passed={demo['passed']} | calls={[c['model'] for c in demo['calls']]} | trace_id={demo['trace_id']}"
)

## 5. Find the step behind the wait

The product question's trace, read back from Elasticsearch: root span, tool, retrieval and the three `chat` spans. Children are not added to their parents.

In [ ]:
provider.force_flush()
SPAN_FIELDS = [
    "name",
    "span_id",
    "parent_span_id",
    "duration",
    "attributes.gen_ai.operation.name",
    "attributes.gen_ai.request.model",
    "attributes.gen_ai.response.model",
    "attributes.gen_ai.usage.input_tokens",
    "attributes.gen_ai.usage.output_tokens",
]


def trace_spans(trace_id, expected=8, timeout=180):
    deadline = time.time() + timeout
    while True:
        es.indices.refresh(index=TRACES, ignore_unavailable=True)
        hits = es.search(
            index=TRACES,
            size=200,
            query={"term": {"trace_id": trace_id}},
            fields=SPAN_FIELDS,
            source=False,
            sort=[{"@timestamp": "asc"}],
            ignore_unavailable=True,
        )["hits"]["hits"]
        if len(hits) >= expected or time.time() > deadline:
            break
        time.sleep(5)
    spans = []
    for hit in hits:
        f = {k: v[0] for k, v in hit.get("fields", {}).items()}
        spans.append(
            {
                "name": f.get("name"),
                "span_id": f.get("span_id"),
                "parent": f.get("parent_span_id"),
                "ms": f.get("duration", 0) / 1e6,
                "op": f.get("attributes.gen_ai.operation.name"),
                "model": f.get("attributes.gen_ai.response.model")
                or f.get("attributes.gen_ai.request.model"),
                "in": f.get("attributes.gen_ai.usage.input_tokens"),
                "out": f.get("attributes.gen_ai.usage.output_tokens"),
            }
        )
    return spans


def breakdown(trace_id):
    spans = trace_spans(trace_id)
    root = next(s for s in spans if s["name"] == "shop.request")
    tool_span = next(s for s in spans if s["name"].startswith("execute_tool"))
    retrieval = next(s for s in spans if s["name"].startswith("retrieve"))
    chats = [s for s in spans if s["op"] == "chat"]
    inner = next(
        (s for s in chats if s["parent"] == tool_span["span_id"]),
        chats[len(chats) // 2],
    )
    outer = [s for s in chats if s is not inner]
    model_ms = sum(s["ms"] for s in chats)
    print(f"root shop.request: {root['ms'] / 1000:.2f} s")
    print(f"{tool_span['name']}: {tool_span['ms'] / 1000:.2f} s")
    print(f"{retrieval['name']}: {retrieval['ms']:.0f} ms")
    for label, s in (
        ("routing chat", outer[0]),
        ("comparison chat", inner),
        ("final chat", outer[-1]),
    ):
        print(
            f"{label} ({s['model']}): {s['ms'] / 1000:.2f} s ({s['ms'] / root['ms']:.1%}) | in={s['in']} out={s['out']}"
        )
    print(
        f"model time: {model_ms / 1000:.2f} s = {model_ms / root['ms']:.1%} of the request"
    )
    print(f"waterfall: {KIBANA_URL}/app/apm/link-to/trace/{trace_id}")


breakdown(demo["trace_id"])

In [ ]:
GENAI_QUERY = f"""FROM {TRACES}
| WHERE resource.attributes.service.name == "{SERVICE}"
    AND attributes.gen_ai.operation.name == "chat"
| STATS calls = COUNT(*), avg_ms = ROUND(AVG(duration) / 1e6),
        input_tokens = SUM(attributes.gen_ai.usage.input_tokens),
        output_tokens = SUM(attributes.gen_ai.usage.output_tokens)
    BY model = attributes.gen_ai.response.model
| SORT avg_ms DESC
| KEEP model, calls, avg_ms, input_tokens, output_tokens
| LIMIT 10"""

esql(GENAI_QUERY, "chat spans by model")

## 6. Attach the model cost to each request

OpenRouter's billed cost per call next to an estimate from the tokens and OpenRouter's public prices, with cached tokens priced separately.

In [ ]:
PRICES = {
    m["id"]: m.get("pricing", {})
    for m in requests.get("https://openrouter.ai/api/v1/models", timeout=60).json()[
        "data"
    ]
}


def cache_tokens(call):
    details = call["usage"].get("input_token_details") or {}
    return details.get("cache_read", 0) or 0, details.get("cache_creation", 0) or 0


def estimate(call):
    price = (
        PRICES.get(call["model"])
        or PRICES.get(call["metadata"].get("model_name") or "")
        or {}
    )
    p = lambda key: float(price.get(key) or 0)
    cache_read, cache_write = cache_tokens(call)
    uncached = (call["usage"].get("input_tokens", 0) or 0) - cache_read - cache_write
    return (
        uncached * p("prompt")
        + cache_read * p("input_cache_read")
        + cache_write * p("input_cache_write")
        + (call["usage"].get("output_tokens", 0) or 0) * p("completion")
    )


for call in demo["calls"]:
    print(
        f"{call['model']}: in={call['usage'].get('input_tokens')} out={call['usage'].get('output_tokens')} "
        f"cache={cache_tokens(call)} | billed ${call['metadata'].get('cost', 0):.6f} | estimate ${estimate(call):.6f}"
    )
print(
    f"app.model_cost.usd = ${demo['cost']:.6f} | estimate ${sum(estimate(c) for c in demo['calls']):.6f}"
)

## 7. Test a lower-priced model on the same questions

Five questions, `REPEATS` times each, one request at a time. Only the model inside `compare_products` changes: Sol for `baseline`, Luna for `candidate`.

In [ ]:
RESULTS = []


def run_variant(variant):
    for repeat in range(1, REPEATS + 1):
        for case, spec in CASES.items():
            r = run_request(spec["question"], variant, case=case)
            RESULTS.append(r)
            print(
                f"{variant:9} {case:8} #{repeat} {r['seconds']:5.2f} s ${r['cost']:.6f} passed={r['passed']}"
                + (f" ERROR {r['error']}" if r["error"] else "")
            )


run_variant("baseline")

In [ ]:
run_variant("candidate")
provider.force_flush()

In [ ]:
EXPERIMENT_QUERY = f"""FROM {TRACES}
| WHERE resource.attributes.service.name == "{SERVICE}"
    AND name == "shop.request"
    AND attributes.app.runset == "{RUNSET}"  # the article uses LIKE "shopping-model-swap*"
| STATS requests = COUNT(*), seconds = ROUND(AVG(duration) / 1e9, 2),
        usd_per_1k = ROUND(AVG(attributes.app.model_cost.usd) * 1000, 3),
        checks = COUNT(*) WHERE attributes.app.check_passed == true
    BY question = attributes.app.case, variant = attributes.app.variant
| SORT question, variant
| KEEP question, variant, requests, checks, seconds, usd_per_1k
| LIMIT 10"""

deadline = time.time() + 180
while time.time() < deadline:
    es.indices.refresh(index=TRACES)
    if sum(r["requests"] for r in esql_rows(EXPERIMENT_QUERY)) >= len(RESULTS):
        break
    time.sleep(5)

esql(EXPERIMENT_QUERY, "baseline vs candidate")
table = {(r["question"], r["variant"]): r for r in esql_rows(EXPERIMENT_QUERY)}
for case in CASES:
    b, c = table.get((case, "baseline")), table.get((case, "candidate"))
    if b and c and b["usd_per_1k"]:
        print(
            f"{case:8} cost {1 - c['usd_per_1k'] / b['usd_per_1k']:.1%} less | latency {b['seconds']:.2f} -> {c['seconds']:.2f} s"
        )

calls = [call for r in RESULTS for call in r["calls"]]
cache_read = sum(cache_tokens(c)[0] for c in calls)
cache_write = sum(cache_tokens(c)[1] for c in calls)
print(
    f"\n{len(RESULTS)} requests | failed checks: {sum(not r['passed'] for r in RESULTS)} | "
    f"billed ${sum(r['cost'] for r in RESULTS):.5f} | model calls: {len(calls)} | "
    f"cache read/write tokens: {cache_read}/{cache_write}"
)

In [ ]:
candidate = next(
    r
    for r in RESULTS
    if r["variant"] == "candidate" and r["case"] == "compact" and not r["error"]
)
print(candidate["answer"], "\n")
breakdown(candidate["trace_id"])

## 8. Check that no prompt or response text was captured

In [ ]:
mine = [
    {"term": {"resource.attributes.service.name": SERVICE}},
    {"term": {"resource.attributes.lab.source": LAB_SOURCE}},
]
for field in [
    "attributes.gen_ai.input.messages",
    "attributes.gen_ai.output.messages",
    "attributes.gen_ai.system_instructions",
    "attributes.gen_ai.usage.output_tokens",
]:
    n = es.count(
        index=TRACES, query={"bool": {"filter": mine + [{"exists": {"field": field}}]}}
    )["count"]
    print(f"spans with {field}: {n}")

## 9. Ask the AI Agent what to optimize next

Paste these prompts into **Agent Builder** (Elastic AI Agent).

In [ ]:
context = f"Service {SERVICE}, trace data stream {TRACES}."
print(f"{KIBANA_URL}/app/agent_builder\n")
print(
    f"{context} Trace {demo['trace_id']}. Inspect this shop-assistant trace. Break down model time and "
    "retrieval time without adding parent and child durations twice. Which call is worth investigating first?\n"
)
print(
    f"{context} Trace {candidate['trace_id']}. In this candidate trace, compare_products returns the final answer "
    "and then the agent calls the model one more time. How long does that last call take, how many output tokens "
    "does it write, and what share of the request is it? Suggest one change to test.\n"
)
print(
    f"{context} Experiment {RUNSET} (root spans named shop.request; attributes app.case, app.variant, "
    "app.check_passed, app.model_cost.usd). Compare baseline and candidate by product question. Which questions "
    "save money without a clear latency improvement? Include request counts and failed checks."
)

## 10. Clean up

Deletes the catalog index and the spans this notebook sent (`lab.source`). The data stream stays.

In [ ]:
provider.shutdown()
es.options(ignore_status=404).indices.delete(index=CATALOG)
deleted = es.options(ignore_status=404).delete_by_query(
    index=TRACES,
    query={
        "bool": {
            "filter": [
                {"term": {"resource.attributes.service.name": SERVICE}},
                {"term": {"resource.attributes.lab.source": LAB_SOURCE}},
            ]
        }
    },
    conflicts="proceed",
    refresh=True,
)
print(f"deleted {CATALOG} and {deleted.get('deleted', 0)} spans")